# 🧠 ReasoningGuard — Proof of Concept on Qwen3.6-27B

**Goal**: prove the OpenInterp ReasoningGuard product works — a probe that scores the *reasoning trace itself* (during thinking) for fabrication risk, distinct from FabricationGuard which scores the prompt.

## The 3 questions we answer

| Question | Metric | Pass threshold |
|---|---|---|
| **Detection works?** | Probe AUROC on reasoning correctness, held-out | ≥ 0.70 |
| **Generalizes across domains?** | Train on GSM8K, eval on MATH / StrategyQA | ≥ 0.65 |
| **Position matters?** | Best (layer, position) outperforms naive end-of-prompt scoring | gap ≥ +0.05 AUROC |

## Foundation

Companion to:
- `30_hallucinationguard_proof_qwen36_27b.ipynb` — single SAE feature (failed cross-bench)
- `31_hallucinationguard_v2_linear_probe.ipynb` — multi-feature LR probe → FabricationGuard (✅ shipped)
- `mechreward` Stage Gate G1 (Qwen3.5-4B): correlation ρ=0.52-0.54 between SAE features and GSM8K correctness — proves residual carries reasoning signal

## What's different vs FabricationGuard

| | FabricationGuard | **ReasoningGuard** |
|---|---|---|
| Score position | Last token of *prompt* (before generation) | Multiple positions during/after `<think>` block |
| Label | Will model fabricate? | Is reasoning trace going to lead to correct answer? |
| Use case | Refuse to answer if model doesn't know entity | Interrupt/restart generation if reasoning derails |
| Latency | 1 score per query | 1 score per ~50-100 reasoning tokens |

## Strategy

1. Generate Qwen3.6-27B *thinking-mode* rollouts on GSM8K + MATH + StrategyQA (~600 questions total)
2. Capture residual at L31 + L55 at **4 positions per response**: end of question, mid-thinking, end of `</think>`, end of answer
3. Label by extracted answer correctness vs ground truth
4. Train LR probe at each (layer, position) combination
5. Identify best — show it beats FabricationGuard's L31/end-of-prompt baseline
6. Cross-domain test: train on GSM8K, evaluate on MATH+StrategyQA held-out
7. Mitigation: simulate "interrupt mode" using probe score during generation

## Hardware

RTX PRO 6000 Blackwell 96 GB on Colab Pro+. Total: ~2-3h, ~R$10-15 in credits.

In [ ]:
!pip -q install --upgrade transformers accelerate safetensors huggingface_hub datasets scipy scikit-learn matplotlib tqdm joblib

## 1. Setup + auth + config

In [ ]:
import os, json, time, math, gc, re
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from huggingface_hub import login, hf_hub_download, HfApi, create_repo

HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN is None:
    import getpass
    HF_TOKEN = getpass.getpass('HF token (write scope): ')
login(HF_TOKEN, add_to_git_credential=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda'
print(f'CUDA: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

CFG = {
    'model':            'Qwen/Qwen3.6-27B',
    'probe_layers':     [31, 40, 55],         # mid + late + paper-grade
    'probe_positions':  ['end_question', 'mid_think', 'end_think', 'end_answer'],
    'subset_size': {
        'gsm8k':       300,
        'math':        200,
        'strategyqa':  150,
    },
    'max_new_tokens':   1024,                 # thinking mode generates a lot
    'train_test_split': 0.7,
    'random_seed':      42,
    'lr_C_sweep':       [0.001, 0.01, 0.1, 1.0, 10.0],
    'hf_results_repo':  os.environ.get('HF_USERNAME', 'caiovicentino1') + '/ReasoningGuard-linearprobe-qwen36-27b',
}
LOCAL_OUT = Path('/content/reasoningguard_out')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
print(json.dumps(CFG, indent=2, default=str))

## 2. Load Qwen3.6-27B (thinking mode capable)

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText, AutoModelForCausalLM

if 'model' not in globals() or 'tok' not in globals():
    print(f'Loading {CFG["model"]} ...')
    tok = AutoTokenizer.from_pretrained(CFG['model'], trust_remote_code=True)
    try:
        model = AutoModelForImageTextToText.from_pretrained(
            CFG['model'], dtype=torch.bfloat16, attn_implementation='sdpa',
            device_map={'': device}, trust_remote_code=True,
        )
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(
            CFG['model'], dtype=torch.bfloat16, attn_implementation='sdpa',
            device_map={'': device}, trust_remote_code=True,
        )
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

def _block_list(m):
    candidates = [m]
    if hasattr(m, 'model'): candidates.append(m.model)
    for s in candidates:
        for path in [('model','language_model','layers'), ('language_model','layers'),
                     ('model','layers'), ('layers',)]:
            cur = s; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                return cur
    raise RuntimeError('layers not found')

blocks = _block_list(model)
d_model = (model.config.text_config.hidden_size if hasattr(model.config, 'text_config')
           else model.config.hidden_size)
print(f'Model: {len(blocks)} layers, d_model = {d_model}')
print(f'GPU mem: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 3. Multi-layer + multi-position residual capture

Hook all probe layers in one pass. For each generated response, capture residuals at 4 positions:
1. **end_question** — last token of original question (before any thinking)
2. **mid_think** — middle of `<think>...</think>` block
3. **end_think** — last token before `</think>` close
4. **end_answer** — last token of full response

In [ ]:
class MultiLayerHook:
    def __init__(self, blocks, layers):
        self.layers = layers
        self.bufs = {l: None for l in layers}
        self.handles = []
        for l in layers:
            self.handles.append(blocks[l].register_forward_hook(self._make(l)))
    def _make(self, l):
        def hook(_mod, _inp, out):
            h = out[0] if isinstance(out, tuple) else out
            self.bufs[l] = h.detach()
        return hook
    def pop(self, layer):
        b = self.bufs[layer]; self.bufs[layer] = None; return b
    def close(self):
        for h in self.handles: h.remove()

ml_hook = MultiLayerHook(blocks, CFG['probe_layers'])

THINK_OPEN = '<think>'
THINK_CLOSE = '</think>'

@torch.no_grad()
def thinking_rollout_with_residuals(question: str, max_new_tokens: int = None):
    """Generate Qwen3.6-27B thinking-mode response + capture residuals at 4 positions.
    
    Returns: dict with 'response', 'positions', 'residuals' (per layer per position).
    """
    max_new_tokens = max_new_tokens or CFG['max_new_tokens']
    messages = [{'role': 'user', 'content': question}]
    prompt_text = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=True,
    )
    enc = tok(prompt_text, return_tensors='pt', truncation=True, max_length=1024).to(device)
    n_prompt = enc['input_ids'].shape[1]
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tok.pad_token_id or tok.eos_token_id,
    )
    full_ids = out[0]
    response_ids = full_ids[n_prompt:]
    response_text = tok.decode(response_ids, skip_special_tokens=False)
    n_total = full_ids.shape[0]
    end_question_pos = n_prompt - 1
    end_answer_pos   = n_total - 1

    # Find <think> + </think> token positions in the response
    think_open_pos  = None
    think_close_pos = None
    
    # Decode incrementally to find <think> and </think>
    decoded_so_far = ''
    for i, tid in enumerate(response_ids.tolist()):
        decoded_so_far = tok.decode(response_ids[:i+1].tolist(), skip_special_tokens=False)
        if think_open_pos is None and THINK_OPEN in decoded_so_far:
            think_open_pos = n_prompt + i
        if think_close_pos is None and THINK_CLOSE in decoded_so_far:
            think_close_pos = n_prompt + i
            break
    
    if think_open_pos is None or think_close_pos is None:
        # No thinking detected — fallback to start/end
        think_open_pos = n_prompt
        think_close_pos = end_answer_pos
    
    mid_think_pos = (think_open_pos + think_close_pos) // 2
    end_think_pos = think_close_pos

    positions = {
        'end_question': end_question_pos,
        'mid_think':    mid_think_pos,
        'end_think':    end_think_pos,
        'end_answer':   end_answer_pos,
    }

    # Run forward pass on full sequence to capture residuals at all layers
    ml_hook.bufs = {l: None for l in CFG['probe_layers']}
    _ = model(full_ids.unsqueeze(0))

    residuals = {}
    for l in CFG['probe_layers']:
        h_layer = ml_hook.pop(l)                                 # (1, T, D) bf16
        if h_layer is None: continue
        per_pos = {}
        for pname, pidx in positions.items():
            pidx_safe = max(0, min(pidx, h_layer.shape[1] - 1))
            per_pos[pname] = h_layer[0, pidx_safe].float().cpu().numpy()
        residuals[l] = per_pos

    return {
        'response': response_text,
        'response_decoded_clean': tok.decode(response_ids, skip_special_tokens=True),
        'positions': positions,
        'residuals': residuals,
        'n_prompt_tokens': n_prompt,
        'n_response_tokens': len(response_ids),
        'has_thinking': think_open_pos is not None and think_close_pos is not None,
    }

# Quick test
r = thinking_rollout_with_residuals('What is 12 × 17?', max_new_tokens=300)
print(f'Response excerpt: {r["response_decoded_clean"][:200]}')
print(f'Positions:        {r["positions"]}')
print(f'Residual shapes:  L31 = {[(k, v.shape) for k,v in r["residuals"][31].items()][:2]}')
print(f'Has thinking:     {r["has_thinking"]}')

## 4. Load reasoning benchmarks

- **GSM8K** — math word problems with numeric answers (canonical math reasoning)
- **MATH** — harder math (competition-level)
- **StrategyQA** — multi-step yes/no reasoning

In [ ]:
from datasets import load_dataset
import random
random.seed(CFG['random_seed'])
np.random.seed(CFG['random_seed'])

def _safe_load(loader, name):
    try: return loader()
    except Exception as e:
        print(f'  [{name}] failed: {e}'); return None

gsm8k = _safe_load(lambda: load_dataset('openai/gsm8k', 'main', split='test'), 'gsm8k')
mathd = _safe_load(lambda: load_dataset('hendrycks/competition_math', split='test'), 'math')
if mathd is None:
    mathd = _safe_load(lambda: load_dataset('lighteval/MATH', 'all', split='test'), 'math-fb')
stratqa = _safe_load(
    lambda: load_dataset('ChilleD/StrategyQA', split='train'), 'strategyqa',
)
if stratqa is None:
    stratqa = _safe_load(
        lambda: load_dataset('voidful/StrategyQA', split='train'), 'strategyqa-fb',
    )

subsets = {}
if gsm8k is not None:
    n = min(CFG['subset_size']['gsm8k'], len(gsm8k))
    subsets['gsm8k'] = [gsm8k[i] for i in random.sample(range(len(gsm8k)), n)]
if mathd is not None:
    n = min(CFG['subset_size']['math'], len(mathd))
    subsets['math'] = [mathd[i] for i in random.sample(range(len(mathd)), n)]
if stratqa is not None:
    n = min(CFG['subset_size']['strategyqa'], len(stratqa))
    subsets['strategyqa'] = [stratqa[i] for i in random.sample(range(len(stratqa)), n)]

print('Subsets:', {k: len(v) for k, v in subsets.items()})

## 5. Answer extraction + grading

Extract numeric answer (GSM8K, MATH) or yes/no (StrategyQA) from generated response.

In [ ]:
ANSWER_PATTERNS = [
    re.compile(r'####\s*(-?\d+(?:\.\d+)?(?:/\d+)?)'),
    re.compile(r'(?:answer|final answer)\s*[:=]?\s*(-?\d+(?:\.\d+)?(?:/\d+)?)', re.IGNORECASE),
    re.compile(r'\\boxed\{(-?\d+(?:\.\d+)?(?:/\d+)?)\}'),
]

def extract_numeric(text):
    for pat in ANSWER_PATTERNS:
        m = pat.search(text)
        if m:
            try:
                v = m.group(1).strip()
                if '/' in v:
                    a, b = v.split('/')
                    return float(a) / float(b)
                return float(v)
            except Exception:
                continue
    nums = re.findall(r'-?\d+(?:\.\d+)?', text)
    if nums: return float(nums[-1])
    return None

def extract_yes_no(text):
    t = text.lower()
    after = t.split('</think>')[-1] if '</think>' in t else t
    if re.search(r'\b(yes|true|correct)\b', after): return True
    if re.search(r'\b(no|false|incorrect)\b', after): return False
    if re.search(r'\b(yes|true)\b', t):  return True
    if re.search(r'\b(no|false)\b', t):  return False
    return None

def grade_gsm8k(generated_text, ground_truth_answer):
    pred = extract_numeric(generated_text)
    truth = extract_numeric(str(ground_truth_answer))
    if pred is None or truth is None: return False
    return abs(pred - truth) < 1e-3

def grade_math(generated_text, ground_truth_solution):
    pred = extract_numeric(generated_text)
    truth = extract_numeric(str(ground_truth_solution))
    if pred is None or truth is None: return False
    return abs(pred - truth) < 1e-3

def grade_strategyqa(generated_text, ground_truth):
    pred = extract_yes_no(generated_text)
    truth = bool(ground_truth) if not isinstance(ground_truth, bool) else ground_truth
    if pred is None: return False
    return pred == truth

# sanity check
print(extract_numeric('The answer is 42.'))            # 42
print(extract_numeric('#### 18'))                       # 18
print(extract_numeric('\\boxed{3.14}'))                 # 3.14
print(extract_yes_no('After analysis, yes.'))           # True

## 6. Generate rollouts with residual capture

**This is the longest step**: ~650 questions × 30s/rollout × thinking mode = ~3-5 hours.

Uses HF checkpointing per-100-questions in case of crash.

In [ ]:
from tqdm.auto import tqdm

data = {bench: {pos: {l: [] for l in CFG['probe_layers']} for pos in CFG['probe_positions']}
        for bench in subsets}
for bench in subsets:
    data[bench]['y']        = []
    data[bench]['response'] = []
    data[bench]['question'] = []

def process_question(question, ground_truth, grade_fn, bench_name):
    try:
        r = thinking_rollout_with_residuals(question)
    except Exception as e:
        print(f'  [{bench_name}] generation failed: {e}'); return
    correct = grade_fn(r['response_decoded_clean'], ground_truth)
    is_hallucinated = int(not correct)
    data[bench_name]['y'].append(is_hallucinated)
    data[bench_name]['response'].append(r['response_decoded_clean'][:1500])
    data[bench_name]['question'].append(question[:300])
    for l in CFG['probe_layers']:
        for pos in CFG['probe_positions']:
            data[bench_name][pos][l].append(r['residuals'].get(l, {}).get(pos, np.zeros(d_model)))

# GSM8K
if 'gsm8k' in subsets:
    for q in tqdm(subsets['gsm8k'], desc='gsm8k rollouts'):
        process_question(q['question'], q['answer'], grade_gsm8k, 'gsm8k')
    print(f'  gsm8k correct rate: {100*(1-np.mean(data["gsm8k"]["y"])):.1f}%')

# MATH
if 'math' in subsets:
    for q in tqdm(subsets['math'], desc='math rollouts'):
        ques = q.get('problem') or q.get('question')
        truth = q.get('solution') or q.get('answer')
        process_question(ques, truth, grade_math, 'math')
    print(f'  math correct rate: {100*(1-np.mean(data["math"]["y"])):.1f}%')

# StrategyQA
if 'strategyqa' in subsets:
    for q in tqdm(subsets['strategyqa'], desc='strategyqa rollouts'):
        ques = q.get('question')
        truth = q.get('answer') if 'answer' in q else q.get('targets')
        if isinstance(truth, list) and truth: truth = truth[0]
        process_question(ques, truth, grade_strategyqa, 'strategyqa')
    print(f'  strategyqa correct rate: {100*(1-np.mean(data["strategyqa"]["y"])):.1f}%')

# Convert to arrays
for bench in data:
    if not data[bench]['y']: continue
    data[bench]['y'] = np.array(data[bench]['y'])
    for pos in CFG['probe_positions']:
        for l in CFG['probe_layers']:
            data[bench][pos][l] = np.array(data[bench][pos][l])
    print(f'  {bench}: N={len(data[bench]["y"])}, hallucination rate {100*data[bench]["y"].mean():.1f}%')

ml_hook.close()

## 7. Train probes per (layer, position) — find best signal

Sweep all 12 combinations of (3 layers × 4 positions) on GSM8K alone. Best combo wins.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split

def train_probe(X_train, y_train, X_test, y_test, C_sweep=CFG['lr_C_sweep']):
    if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
        return None, float('nan'), None, None
    scaler = StandardScaler().fit(X_train)
    cv_n = min(5, np.bincount(y_train).min())
    if cv_n < 2:
        return None, float('nan'), None, None
    cv = StratifiedKFold(n_splits=cv_n, shuffle=True, random_state=CFG['random_seed'])
    clf = LogisticRegressionCV(
        Cs=C_sweep, cv=cv, penalty='l2', solver='lbfgs', max_iter=2000,
        scoring='roc_auc', n_jobs=-1, refit=True,
    )
    clf.fit(scaler.transform(X_train), y_train)
    y_score = clf.predict_proba(scaler.transform(X_test))[:, 1]
    return clf, float(roc_auc_score(y_test, y_score)), scaler, float(clf.C_[0])

# Within-bench split for GSM8K
if 'gsm8k' in data and len(data['gsm8k']['y']) > 30:
    idx = np.arange(len(data['gsm8k']['y']))
    y_all = data['gsm8k']['y']
    if len(np.unique(y_all)) == 2:
        idx_tr, idx_te = train_test_split(idx, test_size=0.3, stratify=y_all, random_state=CFG['random_seed'])
    else:
        idx_tr, idx_te = train_test_split(idx, test_size=0.3, random_state=CFG['random_seed'])
    print(f'GSM8K: train {len(idx_tr)}, test {len(idx_te)}, hallucination rate {y_all.mean()*100:.1f}%')

    sweep_results = {}
    print(f'\n{"layer":>6} {"position":>14} {"AUROC":>8} {"best C":>8}')
    print('-' * 45)
    for l in CFG['probe_layers']:
        for pos in CFG['probe_positions']:
            X = data['gsm8k'][pos][l]
            clf, auroc, scaler, C = train_probe(X[idx_tr], y_all[idx_tr], X[idx_te], y_all[idx_te])
            sweep_results[(l, pos)] = {'auroc': auroc, 'C': C}
            mark = '✅' if auroc >= 0.70 else ('🟡' if auroc >= 0.60 else '❌')
            print(f'{l:>6} {pos:>14} {auroc:>7.3f}  {C if C else "—":>7}  {mark}')

    best_combo = max(sweep_results, key=lambda k: sweep_results[k]['auroc'])
    best_auroc = sweep_results[best_combo]['auroc']
    print(f'\n🎯 Best (layer, position) = L{best_combo[0]} / {best_combo[1]} → AUROC {best_auroc:.3f}')
else:
    print('Not enough GSM8K data — skipping sweep.')
    best_combo = (CFG['probe_layers'][0], CFG['probe_positions'][0])
    best_auroc = float('nan')
    sweep_results = {}

## 8. Cross-domain generalization

Train probe on full GSM8K. Evaluate on held-out MATH + StrategyQA. Pass = AUROC ≥ 0.65 across both.

In [ ]:
best_layer, best_pos = best_combo
Xkey = best_pos

if 'gsm8k' in data and len(data['gsm8k']['y']) > 30:
    X_train_full = data['gsm8k'][Xkey][best_layer]
    y_train_full = data['gsm8k']['y']

    if len(np.unique(y_train_full)) == 2:
        scaler_g = StandardScaler().fit(X_train_full)
        clf_global = LogisticRegressionCV(
            Cs=CFG['lr_C_sweep'], cv=5, penalty='l2', solver='lbfgs',
            max_iter=2000, scoring='roc_auc', n_jobs=-1, refit=True,
        )
        clf_global.fit(scaler_g.transform(X_train_full), y_train_full)

        cross_results = {}
        for bench in ['math', 'strategyqa']:
            if bench not in data or len(data[bench]['y']) < 30: continue
            X_te = data[bench][Xkey][best_layer]
            y_te = data[bench]['y']
            if len(np.unique(y_te)) < 2:
                cross_results[bench] = float('nan'); continue
            scores = clf_global.predict_proba(scaler_g.transform(X_te))[:, 1]
            cross_results[bench] = float(roc_auc_score(y_te, scores))

        print(f'\n=== Cross-domain (trained on GSM8K, layer={best_layer}, pos={best_pos}) ===')
        for b, a in cross_results.items():
            mark = '✅' if a >= 0.65 else ('🟡' if a >= 0.55 else '❌')
            print(f'  {mark} {b:>12} AUROC = {a:.3f}')
        passed_cross = all(a >= 0.65 for a in cross_results.values())
    else:
        passed_cross = False
        cross_results = {}
        print('GSM8K labels are degenerate (all correct or all wrong) — cross test skipped.')
else:
    passed_cross = False
    cross_results = {}

## 9. Compare to FabricationGuard probe (sanity baseline)

Does the reasoning-position probe BEAT the simple end-of-prompt FabricationGuard-style probe? If yes, position matters; if no, the cheap probe is enough.


In [ ]:
if 'gsm8k' in data and (best_layer, 'end_question') in sweep_results:
    fg_baseline = sweep_results[(best_layer, 'end_question')]['auroc']
    rg_score    = sweep_results[(best_layer, best_pos)]['auroc']
    delta = rg_score - fg_baseline
    print(f'FabricationGuard-style (L{best_layer} end_question) AUROC = {fg_baseline:.3f}')
    print(f'ReasoningGuard       (L{best_layer} {best_pos}) AUROC = {rg_score:.3f}')
    print(f'Δ = {delta:+.3f}')
    passed_position = delta >= 0.05
    print(f'\nVerdict: {"✅" if passed_position else "❌"} (need Δ ≥ +0.05)')
else:
    passed_position = False

## 10. Visualization — the killer figure

In [ ]:
import matplotlib.pyplot as plt

if sweep_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Heatmap: layer × position
    ax = axes[0]
    Z = np.zeros((len(CFG['probe_layers']), len(CFG['probe_positions'])))
    for i, l in enumerate(CFG['probe_layers']):
        for j, p in enumerate(CFG['probe_positions']):
            v = sweep_results.get((l, p), {}).get('auroc', 0.5)
            Z[i, j] = v if not np.isnan(v) else 0.5
    im = ax.imshow(Z, cmap='RdYlGn', vmin=0.45, vmax=0.95, aspect='auto')
    ax.set_xticks(range(len(CFG['probe_positions']))); ax.set_xticklabels(CFG['probe_positions'])
    ax.set_yticks(range(len(CFG['probe_layers']))); ax.set_yticklabels([f'L{l}' for l in CFG['probe_layers']])
    ax.set_title(f'AUROC × (layer, position) on GSM8K held-out\nbest = L{best_combo[0]} / {best_combo[1]} → {best_auroc:.3f}')
    for i in range(Z.shape[0]):
        for j in range(Z.shape[1]):
            ax.text(j, i, f'{Z[i,j]:.2f}', ha='center', va='center',
                    color='white' if Z[i,j] < 0.55 else 'black', fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax, label='AUROC')

    # Cross-domain bars
    ax = axes[1]
    if cross_results:
        names = ['gsm8k\n(within)'] + list(cross_results.keys())
        values = [best_auroc] + list(cross_results.values())
        colors = ['#4f46e5'] + ['#10b981' if v >= 0.65 else ('#f59e0b' if v >= 0.55 else '#dc2626') for v in cross_results.values()]
        ax.bar(range(len(names)), values, color=colors, alpha=0.85)
        for i, v in enumerate(values):
            ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
        ax.set_xticks(range(len(names))); ax.set_xticklabels(names)
        ax.axhline(0.65, ls=':', color='gray', alpha=0.5, label='cross-domain target')
        ax.axhline(0.5, ls='--', color='gray', alpha=0.3)
        ax.set_ylim(0.4, 1.0); ax.set_ylabel('AUROC')
        ax.set_title(f'Cross-domain generalization\n(probe trained on GSM8K)')
        ax.legend(loc='lower right', fontsize=9)
        ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    fig.savefig(LOCAL_OUT / 'reasoningguard_headline.png', dpi=200, bbox_inches='tight')
    fig.savefig(LOCAL_OUT / 'reasoningguard_headline.pdf', bbox_inches='tight')
    plt.show()

## 11. Final verdict + save artifacts

In [ ]:
import joblib

passed_detection = best_auroc >= 0.70
n_passed = sum([passed_detection, passed_cross, passed_position])

verdict = {
    'best_layer':            int(best_combo[0]),
    'best_position':         best_combo[1],
    'best_auroc_within':     float(best_auroc),
    'cross_domain_auroc':    cross_results,
    'sweep':                 {f'L{l}_{p}': v for (l,p), v in sweep_results.items()},
    'passed_detection':      bool(passed_detection),
    'passed_cross_domain':   bool(passed_cross),
    'passed_position_gain':  bool(passed_position),
    'cfg':                   CFG,
}
verdict['summary'] = f'{n_passed}/3 thresholds passed'
if n_passed == 3:
    verdict['decision'] = '🟢 GREEN LIGHT — ReasoningGuard ships as v0.3'
elif n_passed >= 1:
    verdict['decision'] = '🟡 PARTIAL — narrow scope or further tuning needed'
else:
    verdict['decision'] = '🔴 PIVOT — reasoning signal does not generalize at this layer/position'

print('=' * 70)
print(f'  ReasoningGuard PoC — {verdict["summary"]}')
print('=' * 70)
print(f'  Detection AUROC (best in-domain):     {best_auroc:.3f}  {"✅" if passed_detection else "❌"}  (need ≥ 0.70)')
if cross_results:
    print(f'  Cross-domain AUROC:                   {"  ".join(f"{b}={a:.3f}" for b,a in cross_results.items())}')
    print(f'  Cross-domain pass:                    {"✅" if passed_cross else "❌"}  (need ≥ 0.65 each)')
if (best_combo[0], 'end_question') in sweep_results:
    delta = best_auroc - sweep_results[(best_combo[0], 'end_question')]['auroc']
    print(f'  Position gain vs end_question:        Δ = {delta:+.3f}  {"✅" if passed_position else "❌"}  (need ≥ +0.05)')
print('=' * 70)
print(f'  {verdict["decision"]}')
print('=' * 70)

(LOCAL_OUT / 'verdict.json').write_text(json.dumps(verdict, indent=2, default=str))

# Save probe artifact
if 'clf_global' in dir() and 'scaler_g' in dir():
    joblib.dump(
        {'probe': clf_global, 'scaler': scaler_g, 'layer': best_layer, 'position': best_pos,
         'threshold': 0.5},
        LOCAL_OUT / 'probe.joblib',
    )
    print(f'Probe saved: {LOCAL_OUT / "probe.joblib"}')

(LOCAL_OUT / 'meta.json').write_text(json.dumps({
    'probe_layer':       f'L{best_layer}',
    'probe_position':    best_pos,
    'd_model':           int(d_model),
    'best_threshold':    0.5,
    'C':                 float(clf_global.C_[0]) if 'clf_global' in dir() else None,
    'training_benchmarks': ['gsm8k'],
    'subset_sizes':      CFG['subset_size'],
    'cross_domain':      cross_results,
}, indent=2, default=str))

# Push to HF
api = HfApi()
create_repo(CFG['hf_results_repo'], exist_ok=True, private=False, token=HF_TOKEN, repo_type='dataset')
api.upload_folder(folder_path=str(LOCAL_OUT), repo_id=CFG['hf_results_repo'], repo_type='dataset', token=HF_TOKEN)
print(f'\n✅ Artifacts pushed to https://huggingface.co/datasets/{CFG["hf_results_repo"]}')

## 12. Where this leaves us

**🟢 GREEN LIGHT (3/3)** → ship `openinterp.ReasoningGuard.from_pretrained()` as v0.3.0:
- Same SDK shape as FabricationGuard (`attach`, `score`, `generate` modes)
- New mode: `interrupt` — stop generation when probe score drops below threshold during reasoning
- Use case: catch fabricated chains-of-thought on R1 / Qwen reasoning / Claude thinking models
- Differentiator from competition: **per-token probe during reasoning** vs LLM-judge after the fact

**🟡 PARTIAL (1-2/3)**:
- If detection but not cross-domain: per-domain probes (math vs strategy vs commonsense)
- If detection but no position gain: ship as is, drop ReasoningGuard branding (it's just FabricationGuard at deeper layer)

**🔴 PIVOT (0/3)**:
- Reasoning correctness isn't linearly encoded in residual at end_question / mid_think / end_think.
- Try: token-level sequence probe (LSTM/transformer over reasoning tokens), or attention-weighted aggregation
- Or: pivot to reasoning-specific signals like saturation detection (mechreward `qwen-efficient`)

## After 🟢

1. Wrap probe.joblib into `openinterp/reasoningguard.py` mirroring `guard.py` (FabricationGuard)
2. Add `openinterp.ReasoningGuard.from_pretrained()` to `__init__.py`
3. Add `openinterp guard --mode reasoning` CLI flag
4. Bump v0.3.0 → PyPI + GitHub release + HF dataset README
5. Add `/products/reasoningguard` route to openinterp.org with same 9-section format
6. Update FabricationGuard landing to mention sister product

## Reading queue

- Notebook 30 + 31 (FabricationGuard PoCs)
- mechreward Stage Gate G1 corpus (Qwen3.5-4B GSM8K + per-token features)
- Goodfire RLFR paper (closest neighbour, but Llama dense)
- DeepSeek-R1 paper for thinking-mode context
- Qwen3.6-27B model card for `enable_thinking` semantics